# CNN / RNN Training - CMPSC 448
Select **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. No Google Drive, Hugging Face login, or pre-generated CSV is required. Run all prepares data and trains all six experiments sequentially.

The first run streams the public Arena dataset (about 1.61 GB) and creates CSVs. Valid local CSVs are reused. `/content` is temporary: a runtime reset deletes local data and results, so download results before disconnecting. Training all six experiments can take several hours and may exceed a Colab session's limits.

In [ ]:
from pathlib import Path
import sys, subprocess, json

REPO_URL = 'https://github.com/cucu071212/CMPSC-448-Midterm.git'
REPO_REF = 'main'  # Branch containing this notebook and training pipeline.
PROJECT_DIR = Path('/content/CMPSC-448-Midterm')
UPDATE_CHECKOUT = False  # Optional git pull; never discards local changes.

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(PROJECT_DIR)], check=True)
elif not (PROJECT_DIR / '.git').is_dir():
    raise RuntimeError(f'{PROJECT_DIR} exists but is not a Git checkout. Choose another PROJECT_DIR.')
if UPDATE_CHECKOUT:
    status = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'status', '--porcelain'], text=True)
    if status.strip():
        raise RuntimeError('Checkout has local changes. Commit or preserve them before enabling UPDATE_CHECKOUT.')
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)

PROJECT_DIR = PROJECT_DIR.resolve()
DATA_DIR = PROJECT_DIR / 'data'
OUTPUT_DIR = PROJECT_DIR / 'results'
sys.path.insert(0, str(PROJECT_DIR))
for name in ('train.py', 'data.py', 'prepare_data.py', 'requirements.txt', 'src/cnn.py', 'src/rnn.py'):
    if not (PROJECT_DIR / name).is_file():
        raise RuntimeError(f'Missing {name}. Use the rnn branch containing this pipeline or update your checkout.')
print('Project:', PROJECT_DIR)
print('Data:', DATA_DIR)
print('Results:', OUTPUT_DIR)

In [ ]:
# Avoid upgrading Colab's preinstalled packages unnecessarily.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(PROJECT_DIR / 'requirements.txt')], check=True)

## Shared Training Settings
All experiments use the same splits and seed. Reduce `BATCH_SIZE` to 32 or 16 if GPU memory is insufficient. Public dataset access was checked on 2026-10-10: it is neither private nor gated; no account or token is needed. If its access policy changes, consult the dataset page and obtain any required approval before retrying.

In [ ]:
import torch

EPOCHS = 15
PATIENCE = 3
BATCH_SIZE = 64
MAX_LEN = 512
SEED = 42
VOCAB_PATHS = {'input': None, 'output': None, 'both': None}
assert torch.cuda.is_available(), 'Select a GPU runtime and rerun this notebook.'
print('GPU:', torch.cuda.get_device_name(0))

# Imports work independently of the notebook's working directory.
from prepare_data import ensure_csvs, validate_csvs
summary = ensure_csvs(DATA_DIR, seed=SEED)
print('CSV validation:', validate_csvs(DATA_DIR))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for mode, vocab_path in VOCAB_PATHS.items():
    if vocab_path is not None:
        assert Path(vocab_path).is_file(), f'Vocabulary file not found: {vocab_path}'
subprocess.run([sys.executable, str(PROJECT_DIR / 'train.py'), '--help'], check=True)

## Shared Training and Result Helpers
The best model is selected using validation Macro-F1, then evaluated once on the test set. Each experiment cell streams its training logs and displays Accuracy / Macro-F1, per-class metrics, training curves, the confusion matrix, and sample predictions. Results are saved under OUTPUT_DIR in a new run folder.

Keep the runtime active during training. Rerunning the helper cell resets the in-memory record of completed experiments; rerun the experiment cells to populate the final comparison again.

In [ ]:
import pandas as pd
from IPython.display import display, Image

EXPERIMENTS = (
    ('cnn', 'output'), ('rnn', 'output'),
    ('cnn', 'input'), ('rnn', 'input'),
    ('cnn', 'both'), ('rnn', 'both'),
)
RUNS = {}

def run_experiment(model, mode, vocab_path=None):
    experiment_dir = OUTPUT_DIR / f'{model}_{mode}'
    before = set(experiment_dir.glob('*/summary.csv'))
    command = [
        sys.executable, '-u', str(PROJECT_DIR / 'train.py'),
        '--model', model, '--mode', mode,
        '--data-dir', str(DATA_DIR), '--output-dir', str(OUTPUT_DIR),
        '--epochs', str(EPOCHS), '--patience', str(PATIENCE),
        '--batch-size', str(BATCH_SIZE), '--max-len', str(MAX_LEN),
        '--seed', str(SEED), '--device', 'cuda', '--num-workers', '0',
    ]
    if vocab_path is not None:
        command += ['--vocab-path', str(vocab_path)]
    print(f'Starting training: {model} / {mode}', flush=True)
    # An argument list handles paths containing spaces without using a shell command string.
    with subprocess.Popen(command, cwd=str(PROJECT_DIR), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    if returncode != 0:
        raise RuntimeError(f'Training failed (exit={returncode}). Check the logs above.')
    created = set(experiment_dir.glob('*/summary.csv')) - before
    assert len(created) == 1, f'Check the results folder for the new run: {experiment_dir}'
    return next(iter(created)).parent

def show_results(run_dir):
    display(pd.read_csv(run_dir / 'summary.csv'))
    metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(metrics['classification_report']).T)
    display(Image(filename=str(run_dir / 'training_and_confusion.png')))
    display(pd.read_csv(run_dir / 'predictions.csv').head())
    print('Saved files:', ', '.join(p.name for p in sorted(run_dir.iterdir())))


def train_and_review(model, mode):
    # Remove any previous result for this experiment before attempting a rerun.
    RUNS.pop((model, mode), None)
    run_dir = run_experiment(model, mode, VOCAB_PATHS[mode])
    RUNS[(model, mode)] = run_dir
    print('Done! Results folder:', run_dir)
    show_results(run_dir)
    return run_dir

## Experiment 1 — CNN: Output Only (RQ1 / RQ2)

In [ ]:
cnn_output_run = train_and_review('cnn', 'output')

## Experiment 2 — RNN: Output Only (RQ1 / RQ2)

In [ ]:
rnn_output_run = train_and_review('rnn', 'output')

## Experiment 3 — CNN: Input Only (RQ2)

In [ ]:
cnn_input_run = train_and_review('cnn', 'input')

## Experiment 4 — RNN: Input Only (RQ2)

In [ ]:
rnn_input_run = train_and_review('rnn', 'input')

## Experiment 5 — CNN: Input + Output (RQ2)

In [ ]:
cnn_both_run = train_and_review('cnn', 'both')

## Experiment 6 — RNN: Input + Output (RQ2)

In [ ]:
rnn_both_run = train_and_review('rnn', 'both')

## Compare All Six Experiments
This cell compares completed experiments recorded in this notebook session and saves `comparison_current.csv`. It lists any missing experiments so a partial table cannot be mistaken for a complete six-experiment comparison. It does not start training or automatically mix in older saved runs.

For RQ1, compare the two output-only rows. For RQ2, compare input, output, and both within each model. Use the results in your report to explain whether prompts improve classification. If input-only performance is high, investigate prompt topic/style differences across labels and duplicate prompts across data splits. Do not select model settings based on test performance.

In [ ]:
tables = []
missing = []
for model, mode in EXPERIMENTS:
    run_dir = RUNS.get((model, mode))
    if run_dir is None:
        missing.append(f'{model} / {mode}')
        continue
    table = pd.read_csv(run_dir / 'summary.csv')
    table['run_dir'] = str(run_dir)
    tables.append(table)

if tables:
    comparison = pd.concat(tables, ignore_index=True)
    display(comparison)
    comparison_path = OUTPUT_DIR / 'comparison_current.csv'
    comparison.to_csv(comparison_path, index=False)
    print('Comparison saved:', comparison_path)
else:
    print('No completed experiments in this session. Run the experiment cells first.')

if missing:
    print(f'Incomplete comparison ({len(tables)}/6 experiments). Missing:', ', '.join(missing))
else:
    print('All six experiments are included.')

## Download Results / Optional Google Drive Backup
A ZIP of `results/` is downloaded by default after training. Browser settings may require allowing the download. You can rerun this cell after any completed experiment. Set `SAVE_TO_DRIVE = True` only if you want a backup of both CSVs and results; this is the only cell that mounts Drive. Existing backup folders are preserved using a timestamp. Runtime resets erase local data and results; the next run recreates CSVs from the pinned public dataset revision.

In [ ]:
import shutil
from datetime import datetime
from IPython.display import FileLink

DOWNLOAD_RESULTS = True
SAVE_TO_DRIVE = False

if any(OUTPUT_DIR.rglob('summary.csv')):
    archive = Path(shutil.make_archive(str(PROJECT_DIR.parent / 'cmpsc448_results'),
                                       'zip', root_dir=OUTPUT_DIR))
    print('Results ZIP:', archive)
    if DOWNLOAD_RESULTS:
        from google.colab import files
        files.download(str(archive))
else:
    print('No completed training results to download yet.')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    backup = Path('/content/drive/MyDrive/CMPSC-448-Midterm') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    backup.mkdir(parents=True, exist_ok=False)
    for directory in (DATA_DIR, OUTPUT_DIR):
        if directory.exists():
            shutil.copytree(directory, backup / directory.name)
    print('Drive backup:', backup)